## Status (rewritten 2026-08-04) -- READ BEFORE RUNNING

**Nothing already computed is recomputed.** Every run cell checks disk first:

- **Harmony**: `SKIP_IF_EXISTS = True` -- a completed (embedding + SEACells + Leiden) run
  for that dataset/dimension is a metrics reload, no clustering re-runs.
- **scProto**: three-way check per dataset. Complete run on disk (checkpoint + metrics +
  umap_cells) -> **skipped entirely**, not even re-evaluated. Checkpoint but missing
  outputs -> reloads the checkpoint and only re-runs evaluation. Nothing on disk ->
  trains.

Pancreas at d=20 was already run (Harmony d=20 + scProto LD20), so both pancreas run
cells below should report **SKIP** and cost seconds. Lung and immune will actually compute.

The d=8 and d=50 runs from the sibling notebooks are read-only here -- they appear in
every table without being touched.

Cell 8 prints exactly what is on disk before anything runs, so you can see in advance
what will compute and what will be reused.

# Harmony at 20 PCs + scProto at a 20-dim latent

**Why 20.** It is the harmony authors' own default: the R package's `HarmonyMatrix` entry
point is `do_pca=TRUE, npcs=20`, and their quickstart vignette uses the top 20 PCs.
(`harmonypy` performs no PCA at all -- it takes whatever embedding you give it. The 50
used in `harmony_default_dim.ipynb` is scanpy's / Seurat's convention, the number Reviewer
nG29 named.) So 20 and 50 are both defensible defaults, from different sources.

**What this adds.** With d=8, d=20 and d=50 on the same datasets we see the shape of the
curve, not two isolated points:

- **Harmony**: is its rare-cell jump between 8 and 50 (pancreas F1 0.32 -> 0.66) gradual,
  or a threshold that 20 already sits above? The embedding-only affinity purity from the
  pancreas d=20 run already hints at the latter (0.525 at d=20 vs. 0.529 at d=50, 0.297 at
  d=8) -- the rare-cell table below is what actually decides it.
- **scProto**: whether anything about it depends on latent dimension. Pancreas so far:
  modularity 0.601 (d=8) / 0.615 (d=20) / 0.611 (d=50) -- flat.

**Metrics reported.** The paper's own: Table 1 (purity, batch entropy, modularity/batch,
coverage), Table 2, and the **rare-cell table** -- coverage, recall, precision,
homogeneity, cross-batch homogeneity and macro F1, each mean +- std across that dataset's
batches -- plus paired one-sided Wilcoxon tests against two references (scProto d=8, the
published model, and scProto d=20, the dimension-matched one).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

In [ ]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [16]:
# Ground truth for "did the install cell above actually work" -- pip's own log is noisy
# (resolver backtracking prints errors for discarded candidate versions even on a fully
# successful install), so importing every package is the real test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {getattr(mod, '__version__', '?')})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)
  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)
  OK   scarches         (import scarches, version 0.6.1)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

All 14 packages import cleanly -- safe to continue.


In [1]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [2]:
import os
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir, get_seacell_model_dir

print("extra imports ready")

extra imports ready


## Config

`CVAE_EPOCHS=50` / `BATCH_SIZE=1024` are the published Stage-1 hyperparameters, unchanged.
`TRAIN_EPOCHS=20` is the short Stage-2 budget -- not the binding constraint: both the d=20
and d=50 pancreas runs early-stopped at epoch 9 with their best modularity at epoch 3.

In [3]:
ALL_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

DIM = 20                 # PCA components for Harmony AND scProto's latent_dims
OTHER_DIMS = [8, 50]     # already on disk from the sibling notebooks -- read-only here

CVAE_EPOCHS = 50
TRAIN_EPOCHS = 20
BATCH_SIZE = 1024
EVAL_FREQ = 3
PATIENCE = 6
UMAP_STEPS_PER_EPOCH = 500

SKIP_IF_EXISTS = True    # Harmony: reload a completed run instead of recomputing it
FORCE_SCPROTO_RERUN = False   # flip to True only to deliberately redo a finished scProto run

CORRECTION_METHODS = ['harmony']
METHOD_DISPLAY_NAMES = {'harmony': f'Harmony d={DIM}'}
RUN_SEACELLS = True
RUN_LEIDEN = True

dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}
REF_DIM_NAME = f'scProto (d={DIM})'

## Helper functions

Harmony goes through `run_all_baselines_for_dataset(..., force_matched_n_comps=DIM)` --
the same shared pipeline every baseline notebook uses. scProto goes through `run_mc_task`,
the entry point that produced the paper's numbers, with `latent_dims` overridden.

The wrappers below add the disk checks (`scproto_status`) and assert that the latent
dimension really took effect, so a silent fallback to 8 can never pass unnoticed.

In [4]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.metric_helpers.result_tables import extract_model_key


def run_harmony_at_dim(ds_id, dim=None):
    """Harmony at `dim` PCs -> {SEACells, Leiden}. Writes to dimension-qualified folders
    (seacell_X_harmony_d{dim} / leiden_X_harmony_d{dim}_K{K}), so other dimensions on disk
    are never touched. With SKIP_IF_EXISTS a finished run is a metrics reload.
    """
    return run_all_baselines_for_dataset(
        ds_id, correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
        run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
        force_matched_n_comps=DIM if dim is None else dim,
    )


def scproto_run_dirs(ds_id, latent_dim):
    """Run folders for scProto trained at `latent_dim`. latent_dims appears in the folder
    name as 'LD{n}' whenever it differs from the default 8 -- that token is what keeps
    these runs separate from the published d=8 ones on disk.
    """
    base = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base):
        return []
    token = f'LD{latent_dim}'
    return sorted(
        d for d in os.listdir(base)
        if token in d and d.startswith('proto_umap') and os.path.isdir(os.path.join(base, d))
    )


def scproto_status(ds_id, latent_dim):
    """What is on disk for this (dataset, latent_dim): 'complete', 'checkpoint_only', or
    'missing', plus the run dir.

    'complete' means checkpoint + metrics.json + umap_cells.csv are all present -- i.e.
    every table in this notebook can already be built from it, so there is nothing to do.
    'checkpoint_only' means the model is trained but its evaluation outputs are missing,
    so evaluation is re-run WITHOUT retraining.
    """
    for d in scproto_run_dirs(ds_id, latent_dim):
        p = os.path.join(get_dataset_model_dir(ds_id), d)
        has_ckpt = os.path.exists(os.path.join(p, 'umap_checkpoint.pth'))
        has_out = (os.path.exists(os.path.join(p, 'metrics.json'))
                   and os.path.exists(os.path.join(p, 'umap_cells.csv')))
        if has_ckpt and has_out:
            return 'complete', p
        if has_ckpt:
            return 'checkpoint_only', p
    return 'missing', None


def train_scproto_at_dim(ds_id, latent_dim=None, force=None):
    """scProto with latent_dims=latent_dim, everything else identical to the published
    configuration (LAMBDA_PROTO_UMAP_PRECON, arbf affinity, the dataset's own
    num_prototypes).

    Does the least work the disk allows: skips completely when the run is finished,
    reloads the checkpoint when only evaluation is missing, trains only when nothing
    exists. Pass force=True to retrain regardless.
    """
    latent_dim = DIM if latent_dim is None else latent_dim
    force = FORCE_SCPROTO_RERUN if force is None else force
    status, path = scproto_status(ds_id, latent_dim)

    if status == 'complete' and not force:
        print(f"=== [{ds_id}] scProto latent_dims={latent_dim}: SKIP -- already complete "
              f"(checkpoint + metrics + umap_cells)\n    {path}")
        return None, None, None

    action = ('training fresh' if status == 'missing' or force
              else 'reloading checkpoint, re-running evaluation only')
    print(f"=== [{ds_id}] scProto latent_dims={latent_dim}: {action} ===")

    t, res, mc = run_mc_task(
        ds_id,
        cvae_epochs=CVAE_EPOCHS, train_epochs=TRAIN_EPOCHS, eval_freq=EVAL_FREQ,
        patience=PATIENCE, batch_size=BATCH_SIZE,
        umap_steps_per_epoch=UMAP_STEPS_PER_EPOCH,
        lambda_config=LAMBDA_PROTO_UMAP_PRECON, affinity_type='arbf',
        load_umap=(status == 'checkpoint_only' and not force),
        trainer_kwargs={'latent_dims': latent_dim},
    )

    # Prove the latent dimension took effect rather than assuming the kwarg propagated:
    # a silent fallback to 8 would invalidate the whole comparison, so fail loudly.
    proto_shape = tuple(t.model.get_prototypes().shape)
    assert t.latent_dims == latent_dim, f"trainer.latent_dims={t.latent_dims}, expected {latent_dim}"
    assert proto_shape[1] == latent_dim, f"prototypes {proto_shape}, expected (K, {latent_dim})"
    assert f'LD{latent_dim}' in os.path.basename(t.get_dump_path()), (
        f"run folder carries no LD{latent_dim} token -- it would collide with another run"
    )
    print(f"[verified] latent_dims={t.latent_dims} | prototypes {proto_shape} | "
          f"run dir: {t.get_dump_path()}")
    return t, res, mc


def scproto_model_key(ds_ids, latent_dim):
    """The single model key every scProto-at-latent_dim run normalizes to (extract_model_key
    strips the dataset/NP/cvae/version tokens, so all datasets share one key and appear as
    one row). None if nothing is on disk yet.
    """
    ds_ids = [ds_ids] if isinstance(ds_ids, str) else ds_ids
    keys = {extract_model_key(d, ds_id=ds)
            for ds in ds_ids for d in scproto_run_dirs(ds, latent_dim)}
    if not keys:
        print(f"No scProto LD{latent_dim} run on disk yet for {ds_ids}.")
        return None
    if len(keys) > 1:
        print(f"WARNING: multiple scProto LD{latent_dim} keys on disk: {sorted(keys)} -- "
              f"using '{sorted(keys)[0]}'.")
    return sorted(keys)[0]


print("helpers ready")

helpers ready


### What is already on disk

Pure read -- lists every (dataset, dimension) run this notebook cares about and whether it
will be reused or computed. Run this first: it is the fastest way to confirm nothing
finished is about to be redone.

In [5]:
rows = []
for ds_id in ALL_DATASETS:
    for d in sorted(set([DIM] + OTHER_DIMS)):
        # Harmony: a finished run has a metrics.json in its SEACells folder
        seacell_dir = get_seacell_model_dir(ds_id, f'X_harmony_d{d}')
        harmony_done = os.path.exists(os.path.join(seacell_dir, 'metrics.json'))
        # scProto: d=8 is the published run (no LD token in its folder name)
        if d == 8:
            sc_status = 'published run (d=8)'
        else:
            sc_status, _ = scproto_status(ds_id, d)
        rows.append({
            'dataset': ds_id, 'dim': d,
            'harmony': 'on disk' if harmony_done else 'MISSING',
            'scproto': sc_status,
            'action_here': ('read-only (other dim)' if d != DIM else
                            ('reuse both' if harmony_done and sc_status == 'complete'
                             else 'will compute')),
        })

pd.DataFrame(rows).set_index(['dataset', 'dim'])

harmony              scproto            action_here
dataset     dim                                                     
pancreas    8    on disk  published run (d=8)  read-only (other dim)
            20   on disk             complete             reuse both
            50   on disk             complete  read-only (other dim)
lung        8    on disk  published run (d=8)  read-only (other dim)
            20   on disk             complete             reuse both
            50   on disk             complete  read-only (other dim)
pbmc-immune 8    on disk  published run (d=8)  read-only (other dim)
            20   MISSING              missing           will compute
            50   on disk             complete  read-only (other dim)

## Runs

One Harmony cell and one scProto cell per dataset. Each is independent and safe to re-run;
a finished run prints SKIP and returns immediately.

**Pancreas is already done** -- both cells should be no-ops.

### Pancreas

In [6]:
pancreas_harmony = run_harmony_at_dim('pancreas')

loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
[pancreas] skipping scProto checkpoint load -- dimension given explicitly (force_matched_n_comps=20) and its affinity-purity row is already saved.

=== [pancreas] batch-correction method: harmony ===
[pancreas] X_harmony_d20: SEACells + Leiden already complete -- skipping affinity graph construction and kNN purity (nothing downstream needs them).
[pancreas] harmony_d20_k100: reusing cached embedding at /content/drive/MyDrive/models/pancreas/_cache_X_harmony_d20_k100_emb.npy
[pancreas] seacell_X_harmony_d20 already computed -- skipping (metrics.json found)
[pancreas] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_pancreas16382_ncomp50_kneighbors50_arbf.pkl (nnz=1171528) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/pancreas/seacell_X_harmony_d20] modularity recomputed against canonical graph: mean_modularity_batch=0.24778033738468602 +/- 0.05991124222332852 (was

 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 1155146 total

  0%|          | 0/16 [00:00<?, ?it/s]

[pancreas] X_pca: affinity purity already saved -- reusing (no graph rebuild).
[pancreas] Raw PCA (uncorrected) (d=50): 0.385 +/- 0.164 (n=8 batches)
[pancreas] X_harmony_d20: affinity purity already saved -- reusing (no graph rebuild).
[pancreas] Harmony (d=20): 0.525 +/- 0.177 (n=8 batches)
[pancreas] X_scproto: affinity purity already saved -- reusing (no graph rebuild).
[pancreas] scProto (d=8): 0.454 +/- 0.184 (n=8 batches)
[pancreas] affinity purity saved to /content/drive/MyDrive/models/pancreas/rare_affinity_purity_pancreas.json (merged with any existing entries from other runs/notebooks)


In [7]:
t_panc, res_panc, mc_panc = train_scproto_at_dim('pancreas')
res_panc

=== [pancreas] scProto latent_dims=20: SKIP -- already complete (checkpoint + metrics + umap_cells)
    /content/drive/MyDrive/models/pancreas/proto_umap_ds-panc_NP220_LD20_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31


### Lung

In [8]:
lung_harmony = run_harmony_at_dim('lung')

loading lung data
✅ Already subsetted to HVGs (4000 genes).
[lung] skipping scProto checkpoint load -- dimension given explicitly (force_matched_n_comps=20) and its affinity-purity row is already saved.

=== [lung] batch-correction method: harmony ===
[lung] X_harmony_d20: SEACells + Leiden already complete -- skipping affinity graph construction and kNN purity (nothing downstream needs them).
[lung] harmony_d20_k100: reusing cached embedding at /content/drive/MyDrive/models/lung/_cache_X_harmony_d20_k100_emb.npy
[lung] seacell_X_harmony_d20 already computed -- skipping (metrics.json found)
[lung] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_lung32472_ncomp50_kneighbors50_arbf.pkl (nnz=2480396) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/lung/seacell_X_harmony_d20] modularity recomputed against canonical graph: mean_modularity_batch=0.3005427131212899 +/- 0.072030664634303 (was 0.3005427131212899), K_target=300
[lung] l

  0%|          | 0/32 [00:00<?, ?it/s]

[lung] X_pca: affinity purity already saved -- reusing (no graph rebuild).
[lung] Raw PCA (uncorrected) (d=50): 0.559 +/- 0.194 (n=15 batches)
[lung] X_harmony_d20: affinity purity already saved -- reusing (no graph rebuild).
[lung] Harmony (d=20): 0.571 +/- 0.128 (n=15 batches)
[lung] X_scproto: affinity purity already saved -- reusing (no graph rebuild).
[lung] scProto (d=8): 0.586 +/- 0.208 (n=15 batches)
[lung] affinity purity saved to /content/drive/MyDrive/models/lung/rare_affinity_purity_lung.json (merged with any existing entries from other runs/notebooks)


In [9]:
t_lung, res_lung, mc_lung = train_scproto_at_dim('lung')
res_lung

=== [lung] scProto latent_dims=20: SKIP -- already complete (checkpoint + metrics + umap_cells)
    /content/drive/MyDrive/models/lung/proto_umap_ds-lung_LD20_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31


### Immune (PBMC)

In [ ]:
immune_harmony = run_harmony_at_dim('pbmc-immune')

loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
[pbmc-immune] skipping scProto checkpoint load -- dimension given explicitly (force_matched_n_comps=20) and its affinity-purity row is already saved.

=== [pbmc-immune] batch-correction method: harmony ===


2026-08-04 10:47:23,054 - harmonypy - INFO - Running Harmony
INFO:harmonypy:Running Harmony
2026-08-04 10:47:23,056 - harmonypy - INFO -   Parameters:
INFO:harmonypy:  Parameters:
2026-08-04 10:47:23,058 - harmonypy - INFO -     max_iter_harmony: 10
INFO:harmonypy:    max_iter_harmony: 10
2026-08-04 10:47:23,060 - harmonypy - INFO -     max_iter_kmeans: 4
INFO:harmonypy:    max_iter_kmeans: 4
2026-08-04 10:47:23,061 - harmonypy - INFO -     epsilon_cluster: 0.001
INFO:harmonypy:    epsilon_cluster: 0.001
2026-08-04 10:47:23,062 - harmonypy - INFO -     epsilon_harmony: 0.01
INFO:harmonypy:    epsilon_harmony: 0.01
2026-08-04 10:47:23,064 - harmonypy - INFO -     nclust: 100
INFO:harmonypy:    nclust: 100
2026-08-04 10:47:23,066 - harmonypy - INFO -     block_size: 0.05
INFO:harmonypy:    block_size: 0.05
2026-08-04 10:47:23,067 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
INFO:harmonypy:    lamb: dynamic (alpha=0.2)
2026-08-04 10:47:23,070 - harmonypy - INFO -     theta: [2. 2. 2

[pbmc-immune] harmony_d20_k100: cached embedding to /content/drive/MyDrive/models/pbmc-immune/_cache_X_harmony_d20_k100_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Randomly initialized A matrix.
Setting convergence threshold at 286.24514
Starting iteration 1.
Completed iteration 1.
Starting iteration 10.
Completed iteration 10.
Converged after 16 iterations.


100%|██████████| 300/300 [00:02<00:00, 104.38it/s]


saving to:  /content/drive/MyDrive/models/pbmc-immune/seacell_X_harmony_d20
  delta kept: X=no (deduped), 1 layer(s), 0 varm, 2 obsm, 2 obsp, obs cols ['SEACell']
  saved soft_assignments.npz (33506, 300) to /content/drive/MyDrive/models/pbmc-immune/seacell_X_harmony_d20
Loading SEACell from /content/drive/MyDrive/models/pbmc-immune/seacell_X_harmony_d20 ...
[seacell] unused protos: 0/300 (0.00%)
[seacell] mean cell-type purity: 0.8833  (size-weighted: 0.8782 ± 0.1349)
[seacell] mean batch entropy: 0.5973  (size-weighted: 0.8381 ± 0.4282)
[seacell] coverage: 1.0000
[seacell] modularity: 0.4888
[seacell] per-batch modularity: mean=0.4591, std=0.0261
[aff_dc_compactness] looking for graph at: ./graphs/affinity_pbmc-immune33506_ncomp50_kneighbors50_arbf.pkl
[aff_dc_compactness] mean=1.7593 | saved to /content/drive/MyDrive/models/pbmc-immune/seacell_X_harmony_d20/aff_dc_compactness.csv
[seacell] saved metrics to /content/drive/MyDrive/models/pbmc-immune/seacell_X_harmony_d20
SEACell UMAP 

  0%|          | 0/5 [00:00<?, ?it/s]

Deleted: tmp_c51fbf5e.h5ad
[seacell task2] coverage: 1.0000
[seacell task2] scgraph_corr_avg: 0.8418
[seacell task2] scgraph_corr_std: 0.0708
  [leiden oversegment] resolution=1.0000 -> 16 clusters (need >= 300)
  [leiden oversegment] resolution=2.0000 -> 26 clusters (need >= 300)
  [leiden oversegment] resolution=4.0000 -> 40 clusters (need >= 300)
  [leiden oversegment] resolution=8.0000 -> 75 clusters (need >= 300)
  [leiden oversegment] resolution=16.0000 -> 158 clusters (need >= 300)


In [ ]:
t_imm, res_imm, mc_imm = train_scproto_at_dim('pbmc-immune')
res_imm

## Results

Everything below is a pure read of saved files -- no training, no clustering. Rows: scProto
at d=8 (published), d=20 and d=50; Harmony at d=8, d=20 and d=50, each with SEACells and
Leiden on top; SEACells (PCA) as the paper's own baseline. A row whose run is not on disk
is simply absent.

In [ ]:
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, dim_matched_read_only_keywords, render_full_comparison_report,
    RARE_CELL_METRICS, RARE_CELL_SIG_METRICS,
)
from interpretable_ssl.evaluation.paper_figures import (
    rare_metric_significance_paired, graph_batch_significance_paired,
)

# Harmony at DIM plus the other dimensions read off disk. Distinct display names per
# dimension -- identical names would collapse into one row and silently drop a dimension.
read_only = {}
for d in OTHER_DIMS:
    read_only.update(dim_matched_read_only_keywords('harmony', f'Harmony d={d}', matched_dim=d))

MODEL_KEYWORDS = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES, matched_dim=DIM, extra_read_only=read_only,
)
for d in [DIM] + OTHER_DIMS:
    if d == 8:
        continue   # the published d=8 scProto run is already in MODEL_KEYWORDS as 'scProto'
    key = scproto_model_key(ALL_DATASETS, d)
    if key:
        MODEL_KEYWORDS[key] = f'scProto (d={d})'

MODEL_KEYWORDS

In [ ]:
# Table 1, Table 2, the rare-cell table, and all four significance tests, with scProto
# (published, d=8) as reference.
report = render_full_comparison_report(
    ALL_DATASETS, dataset_display_names, MODEL_KEYWORDS, ref_name='scProto',
)

### Rare-cell metrics on their own

Same table as inside the report above, isolated for reading: coverage, recall, precision,
homogeneity, cross-batch homogeneity and macro F1, mean +- std across batches.

Read F1 together with its two halves -- precision and recall move in opposite directions
between scProto and Harmony, and cross-batch homogeneity is the metric Harmony's
batch-mixing objective most directly targets.

In [ ]:
show_table(report['rare'], metrics=RARE_CELL_METRICS,
           dataset_display_names=dataset_display_names)

In [ ]:
# Same-dimension comparison: scProto (d=20) as reference, against Harmony d=20.
# Reuses the rare table already computed above.
sig_rare = rare_metric_significance_paired(
    report['rare'], ref_name=REF_DIM_NAME, metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
print(f"=== Rare-cell metrics, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other), "
      f"Bonferroni-corrected per dataset ===")
display(sig_rare)

sig_mod = graph_batch_significance_paired(
    ALL_DATASETS, MODEL_KEYWORDS, ref_name=REF_DIM_NAME,
    dataset_display_names=dataset_display_names,
)
print(f"\n=== Modularity per batch, PAIRED one-sided Wilcoxon ({REF_DIM_NAME} > other) ===")
display(sig_mod)

### Batch entropy vs. latent dimension

Per-metacell batch entropy straight from each run's saved `batch_entropy_per_mc.csv`.
`frac_single_batch` is the share of metacells containing cells from exactly one batch --
what a median of 0 is really saying. This is the check on whether scProto's latent
dimension affects how much batch structure survives into its metacells.

In [ ]:
import numpy as np
from interpretable_ssl.evaluation.paper_figures import _resolve_run_dir, _read_series


def batch_entropy_table(ds_ids, model_keywords):
    ds_ids = [ds_ids] if isinstance(ds_ids, str) else ds_ids
    rows = []
    for ds_id in ds_ids:
        for keyword, name in model_keywords.items():
            run_dir = _resolve_run_dir(ds_id, keyword, prefer_csv='batch_entropy_per_mc.csv')
            if run_dir is None:
                continue
            ser = _read_series(os.path.join(run_dir, 'batch_entropy_per_mc.csv'))
            if ser is None:
                continue
            v = ser.values.astype(float)
            rows.append({
                'dataset': dataset_display_names.get(ds_id, ds_id), 'method': name,
                'n_metacells': len(v),
                'entropy_mean': round(float(np.mean(v)), 3),
                'entropy_median': round(float(np.median(v)), 3),
                'frac_single_batch': round(float((v <= 1e-9).mean()), 3),
            })
    return pd.DataFrame(rows).sort_values(['dataset', 'entropy_median'], ascending=[True, False])


batch_entropy_table(ALL_DATASETS, MODEL_KEYWORDS)

### Realized cluster/metacell count vs. K

Confirms Harmony's downstream SEACells/Leiden landed at each dataset's `num_prototypes`,
so every method is compared at equal K.

In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

harmony_tag = f'X_harmony_d{DIM}'
target_k = {ds: DATASETS[ds]['num_prototypes'] for ds in ALL_DATASETS}

df_k = load_task1_multi(ALL_DATASETS, metrics=['n_clusters', 'resolution'])
if not df_k.empty:
    is_leiden = df_k.index.get_level_values('run').str.startswith(f'leiden_{harmony_tag}')
    if is_leiden.any():
        out = df_k[is_leiden].copy()
        out['target_k'] = [target_k[ds] for ds, _run in out.index]
        out['matches_target'] = out['n_clusters'] == out['target_k']
        display(out)
    else:
        print(f"No leiden_{harmony_tag} run found yet.")

rows = []
for ds_id in ALL_DATASETS:
    n_actual = get_realized_seacell_count(ds_id, harmony_tag)
    k = target_k[ds_id]
    rows.append({'dataset': ds_id, 'method': harmony_tag, 'n_actual': n_actual, 'target_k': k,
                 'matches_target': n_actual is not None and abs(n_actual - k) <= 0.05 * k})
display(pd.DataFrame(rows).set_index(['dataset', 'method']))

### Embedding-only rare-cell affinity purity (no clustering)

One ARBF affinity graph built directly on each embedding; per locally-rare-type cell, the
fraction of its total affinity mass going to same-type cells. Computed automatically
inside the Harmony cells above; this only loads the saved results. The `dim` column
separates the d=8 / d=20 / d=50 Harmony rows.

In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

load_and_compare_affinity_purity(ALL_DATASETS, dataset_display_names=dataset_display_names)

In [ ]:
# Flat CSV of the rare-cell table -- convenient for pasting numbers into the write-up.
save_path = '/content/drive/MyDrive/models/rare_metrics_harmony_d20_all.csv'
report['rare'].to_csv(save_path)
print(f"saved to {save_path}")
report['rare']